In [ ]:
import sys, os
sys.path.append('..')
import numpy as np
import tenpy 
import utils.io as io
import src.observables as obs
DATA_DIR = '../../data/iDMRG'

In [ ]:
loadfile = os.path.join(DATA_DIR, 'rev/scanChi200/Vpm_4.350_chi200.h5')
psi, results = io.load_mps_with_metadata(loadfile)
print("correlation length = ", psi.correlation_length2())
diags = results["diagnostics"]
diags.keys()

In [ ]:
import numpy as np


def block_entropy_slope(
    psi,
    n_values=None,
    first_site=0,
    fit_start=0,
    return_data=False,
):
    """
    Extract the finite-block entropy scaling coefficient a_n from an iMPS.

    Fits
        S(n) = a_n * ln(n) + b

    Parameters
    ----------
    psi : tenpy.networks.mps.MPS
        Converged infinite MPS.

    n_values : iterable of int, optional
        Block sizes to use. For an iMPS, these are contiguous blocks
        [first_site, ..., first_site + n - 1].
        Default: 2, 3, ..., 12.

    first_site : int
        Starting site of the block.

    fit_start : int
        Index into n_values. Points before this index are discarded
        from the linear fit. This is useful for removing small-n
        finite-block corrections.

    return_data : bool
        If True, return (a_n, intercept, n_values, S_values).
        Otherwise return only a_n.

    Returns
    -------
    a_n : float
        Slope in
            S(n) = a_n * ln(n) + const

    intercept : float, optional
        Returned only if return_data=True.

    n_values : ndarray, optional
        Block sizes actually used.

    S_values : ndarray, optional
        Corresponding block entropies.
    """
    if n_values is None:
        n_values = np.arange(2, 13)
    else:
        n_values = np.asarray(list(n_values), dtype=int)

    if np.any(n_values < 1):
        raise ValueError("All block sizes must be positive integers.")

    if fit_start < 0 or fit_start >= len(n_values):
        raise ValueError("fit_start must be between 0 and len(n_values)-1.")

    S_values = []

    for n in n_values:
        segment = list(range(first_site, first_site + n))

        # TeNPy explicitly constructs rho_A for this block and
        # computes its von-Neumann entropy.
        S = psi.entanglement_entropy_segment(
            segment=segment,
            n=1,
        )[0]

        S_values.append(S)

    S_values = np.asarray(S_values)

    # Only use the asymptotic part of the block-size data.
    n_fit = n_values[fit_start:]
    S_fit = S_values[fit_start:]

    x = np.log(n_fit)

    # S(n) = a_n * ln(n) + b
    a_n, intercept = np.polyfit(x, S_fit, 1)

    if return_data:
        return a_n, intercept, n_values, S_values

    return a_n